# 08 Benchmark w/ PNNL & reality
By Tahreem Karim, Chalse Okorom

*Where do our predictions agree with PNNL's projected sites and real data centers?* PNNL's IM3 projections from CERF-DC are from a deterministic cost minimizer equation that never considers observed sites
or policy. Therefore, it makes sense to see where the projections from our model lines up with their equation's projections. We’re also testing our model against real data center sites, so we use the same real sites to check how PNNL’s equation performs, independently (so it's **us vs. reality, us vs. PNNL, and PNNL vs. reality**). Note, the PNNL projections are a **benchmark**, not ground truth. When this analysis determines that the PNNL equation and the Milestone 2 model are in:
- **agreement:** it means two different approaches point at the same counties
- **disagreement:** approaches differ

1. Place every projected campus (20 scenarios) in the county containing centroid (Census 2024 boundaries).
2. Reference scenario for head to head maps: **moderate growth, 50% market gravity** (328 sites), as on the dashboard.
3. Within the states PNNL modeled, compare PNNL's chosen counties with our top ranked counties
   (same number of counties), and w/ CERF's (own) suitability ranking.

##### Outputs `processed/stage6_pnnl/pnnl_county.csv` (one row per county) and `pnnl_comparison.csv`.

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from shapely import wkt
from sklearn.metrics import average_precision_score

from pipeline_config import GROUP, ROOT, STAGE2, STAGE4, STAGE6, TARGET, load_model_table

REF = ("moderate_growth", 50)
d = load_model_table()

## Projected data centers to counties
This cell takes PNNL’s projected future sites (called 'campuses') & figures out which county each one lands in, so they can be compared against our model’s county level predictions.

In [ ]:
files = sorted((STAGE2 / "im3_csv").glob("*_market_gravity.csv"))
sites = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
geom = gpd.GeoSeries(sites.geometry_wkt.map(wkt.loads), crs="ESRI:102003")
pts = gpd.GeoDataFrame(sites.drop(columns="geometry_wkt"), geometry=geom.centroid, crs="ESRI:102003")
counties = gpd.read_file(ROOT / "raw data" / "census" / "cb_2024_us_county_500k")[["GEOID", "STUSPS", "geometry"]]
pts = gpd.sjoin(pts.to_crs(counties.crs), counties, how="left", predicate="within").drop(columns="index_right")

# Check
assert len(pts) == len(sites) == 16645, "a site matched two counties or the file count changed"
print(f"{len(sites):,} projected sites in {len(files)} scenarios; {pts.GEOID.isna().sum()} outside every county")

pts["county_fips"] = pts.GEOID
modeled_states = sorted(pts.STUSPS.dropna().unique())
by_scen = pts.groupby(["county_fips", "growth_scenario", "market_gravity_weight"]).size().rename("n").reset_index()
ref_sites = pts[(pts.growth_scenario == REF[0]) & (pts.market_gravity_weight == REF[1])]

county = pd.DataFrame({
    "n_projected_ref": ref_sites.groupby("county_fips").size(),
    "it_power_mw_ref": ref_sites.groupby("county_fips").data_center_it_power_mw.sum(),
    "n_scenarios_sited": by_scen.groupby("county_fips").size()}).fillna(0)

# Check
print(f"PNNL modeled {len(modeled_states)} states: {' '.join(modeled_states)}")
print(f"reference scenario: {len(ref_sites)} sites in {(county.n_projected_ref > 0).sum()} counties")

16,645 projected sites in 20 scenarios; 0 outside every county
PNNL modeled 37 states: AL AZ CA CO CT FL GA IA ID IL IN KY MA MD MI MN MO MT NC ND NE NJ NM NV NY OH OK OR PA SC TN TX UT VA WA WI WY
reference scenario: 328 sites in 72 counties


## Join to "out of fold" predictions

Probabilities are ntbk 06's calibrated out of fold predictions (**XGBoost**), averaged over three seeds, so
no county is scored by a model that trained on its state. Each county gets one row w/ three probability columns.

In [ ]:
oof = pd.read_csv(STAGE4 / "oof_predictions.csv", dtype={"county_fips": str})
probs = oof[oof.model == "XGBoost"].pivot_table(index="county_fips", columns="set", values="prob", aggfunc="mean")
probs.columns = ["p_" + {"Infrastructure": "infra", "+ Geography": "geo", "+ AI policy": "policy"}[c] for c in probs.columns]

t = d.set_index("county_fips").join(probs).join(county).fillna({"n_projected_ref": 0, "n_scenarios_sited": 0, "it_power_mw_ref": 0})
t["pnnl_modeled"] = t[GROUP].isin(modeled_states)
t["pnnl_ref"] = (t.n_projected_ref > 0).astype(int)
t[["pnnl_modeled", "pnnl_ref", "n_projected_ref", "it_power_mw_ref", "n_scenarios_sited"]].to_csv(STAGE6 / "pnnl_county.csv")

m = t[t.pnnl_modeled]
K = int(m.pnnl_ref.sum())

# Check
print(f"{len(m):,} of our counties are in PNNL-modeled states; PNNL's reference scenario picks {K} of them")

2,604 of our counties are in PNNL-modeled states; PNNL's reference scenario picks 72 of them


## Agreement

**Top K overlap:** our K highest probability counties against PNNL's K chosen counties (same states).
Random choice would overlap by K²/N on average. **PR-AUC vs PNNL picks** treats PNNL's choice as the label.

In [ ]:
def compare(score, name):
    top = set(m[score].sort_values(ascending=False).index[:K])
    pick = set(m.index[m.pnnl_ref == 1])
    both = len(top & pick)
    rho = spearmanr(m[score], m.n_scenarios_sited).statistic
    return {"ranking": name, "overlap": both, "of": K, "jaccard": both / len(top | pick),
            "expected_if_random": K * K / len(m), "pr_auc_vs_pnnl_ref": average_precision_score(m.pnnl_ref, m[score]),
            "spearman_vs_scenarios_sited": rho}


rows = [compare("p_infra", "Ours: infrastructure only"), compare("p_geo", "Ours: + geography"),
        compare("p_policy", "Ours: + AI policy"),
        compare("composite_siting_suitability_mean", "CERF composite suitability (no learning)"),
        compare("atlas_record_count", "Existing atlas facilities (count)")]

comparison = pd.DataFrame(rows)
comparison.to_csv(STAGE6 / "pnnl_comparison.csv", index=False) # Output
comparison.round(3)

,ranking,overlap,of,jaccard,expected_if_random,pr_auc_vs_pnnl_ref,spearman_vs_scenarios_sited
0,Ours: infrastructure only,13,72,0.099,1.991,0.157,0.172
1,Ours: + geography,15,72,0.116,1.991,0.145,0.191
2,Ours: + AI policy,15,72,0.116,1.991,0.122,0.164
3,CERF composite suitability (no learning),9,72,0.067,1.991,0.092,0.241
4,Existing atlas facilities (count),32,72,0.286,1.991,0.365,0.273


>All three versions of our model agree w/ PNNL’s picks! More than chance would predict (13-15 overlapping counties out of 72 vs ~2 expected by chance). But more features we add, the less we agree w/ PNNL (PR-AUC vs. PNNL drops from 0.157 → 0.145 → 0.122 as geography, then policy are added). *Unfortunately counting existing data center facilities still beats every model (0.365 PR-AUC, 32/72 overlap) including PNNL's equation.*

## What PNNL picks & where we disagree

In [ ]:
pick = m[m.pnnl_ref == 1]
print(f"PNNL reference counties with an existing atlas facility: {pick[TARGET].mean():.1%} "
      f"(vs {m[TARGET].mean():.1%} of modeled counties)")
print(f"PNNL reference sites by state (top 6): {ref_sites.STUSPS.value_counts().head(6).to_dict()}")

top_ours = set(m.p_policy.sort_values(ascending=False).index[:K])
only_pnnl = m[(m.pnnl_ref == 1) & ~m.index.isin(top_ours)]
only_ours = m[(m.pnnl_ref == 0) & m.index.isin(top_ours)]
cols = ["county_name", GROUP, TARGET, "p_policy", "n_projected_ref", "n_scenarios_sited",
        "land_value_dollar_per_sqft_mean", "composite_siting_suitability_mean", "population_2024"]

summary = pd.DataFrame({
    "PNNL only": only_pnnl[cols[2:]].median(numeric_only=True),
    "Ours only": only_ours[cols[2:]].median(numeric_only=True),
    "Both": m[(m.pnnl_ref == 1) & m.index.isin(top_ours)][cols[2:]].median(numeric_only=True)})
summary.to_csv(STAGE6 / "disagreement_medians.csv") # Output
summary.round(3)

PNNL reference counties with an existing atlas facility: 66.7% (vs 8.9% of modeled counties)
PNNL reference sites by state (top 6): {'VA': 76, 'TX': 49, 'CA': 21, 'IL': 16, 'AZ': 14, 'GA': 14}


,PNNL only,Ours only,Both
has_data_center,1.000,0.000,1.000
p_policy,0.151,0.562,0.545
n_projected_ref,2.000,0.000,5.000
n_scenarios_sited,13.000,0.000,16.000
land_value_dollar_per_sqft_mean,0.137,0.313,0.321
composite_siting_suitability_mean,0.014,0.011,0.021
population_2024,159285.000,594160.000,1003869.000


>Disagreement splits cleanly along “already has infrastructure” vs. “has favorable policy.” PNNL’s unique picks are counties w/ an existing data center (100% of the time), smaller (median pop. ~159K), w/ low policy scores (0.151) in places PNNL’s equation is confident about (sited in 13/20 scenarios). Our unique picks are the opposite, no existing data center, bigger/more urban (median pop ~594K), and high policy scores (0.562).

### Why we miss PNNL’s steady picks

In [6]:
only_pnnl.sort_values("n_projected_ref", ascending=False)[cols].head(10).round(3)

,county_name,state_abbr,has_data_center,p_policy,n_projected_ref,n_scenarios_sited,land_value_dollar_per_sqft_mean,composite_siting_suitability_mean,population_2024
county_fips,,,,,,,,,
51117,MECKLENBURG,VA,1,0.023,21.0,18.0,0.060,0.008,30824
48113,DALLAS,TX,1,0.437,13.0,16.0,0.155,0.046,2656028
51107,LOUDOUN,VA,1,0.185,13.0,17.0,0.225,0.006,443380
51137,ORANGE,VA,0,0.063,12.0,17.0,0.121,0.017,38986
53025,GRANT,WA,1,0.050,11.0,20.0,0.096,0.016,104717
48029,BEXAR,TX,1,0.300,11.0,16.0,0.138,0.032,2127737
42055,FRANKLIN,PA,0,0.286,10.0,17.0,0.163,0.030,159285
19153,POLK,IA,1,0.246,9.0,16.0,0.178,0.034,516185
04025,YAVAPAI,AZ,0,0.023,8.0,12.0,0.043,0.001,252013


>Some places like Loudoun County, VA (Data Center Alley) already have real data centers (`has_data_center=1`) but have low policy scores (sub 0.05). This might be what’s dragging them out of our top 72, even though they’re some of the strongest, most consistently chosen markets. So right now, relying on policy means underranking a few big, established hubs. A clear future direction to investigate.